# AI-Text Humanizer — Modeling Notebook

Loads the dataset from your **local project folder** (already copied from the HF cache) and builds two models:

1. **Detector (baseline oracle)** — TF-IDF + Logistic Regression classifier distinguishing human vs AI text. This becomes your evaluation metric later: a good humanizer should push the detector's confidence toward "human."
2. **Humanizer (seq2seq)** — T5-small fine-tuning skeleton for AI → human style transfer.

**Sections:**
1. Setup
2. Load dataset locally
3. Quick recap EDA
4. Baseline detector (TF-IDF + Logistic Regression)
5. Detector v2 (fine-tuned DistilBERT) — optional, GPU-recommended
6. Humanizer skeleton (T5 seq2seq)
7. Evaluation loop (detector-as-oracle)
8. Save artifacts


## 1. Setup

In [44]:
#pip install transformers datasets scikit-learn torch accelerate sentencepiece --break-system-packages

In [45]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)


## 2. Load Dataset From Local Folder

This auto-detects your copied dataset folder (matches anything starting with `Rajarshi-Roy-research`) and recursively finds the `.parquet` files inside it — this works whether you copied the raw HF cache structure (`snapshots/<hash>/*.parquet`) or a flattened export.

**If auto-detection fails**, set `DATASET_DIR` manually to the exact path shown in your file explorer.


In [46]:
# Auto-detect the dataset folder (searches current dir and one level down)
candidates = glob.glob("Rajarshi-Roy-research*") + glob.glob("*/Rajarshi-Roy-research*")

if candidates:
    DATASET_DIR = candidates[0]
    print("Found dataset folder:", DATASET_DIR)
else:
    # Fallback: set this manually to match your file explorer path, e.g.:
    # DATASET_DIR = r"Rajarshi-Roy-research__defactify_text_dataset"
    DATASET_DIR = None
    print("Could not auto-detect. Set DATASET_DIR manually in this cell.")


Found dataset folder: Rajarshi-Roy-research___defactify_text_dataset


In [47]:
data_files = (
    glob.glob(os.path.join(DATASET_DIR, "**", "*.arrow"), recursive=True)
    + glob.glob(os.path.join(DATASET_DIR, "**", "*.parquet"), recursive=True)
)
print(f"Found {len(data_files)} data file(s):")
for f in data_files:
    print(" -", f)

Found 3 data file(s):
 - Rajarshi-Roy-research___defactify_text_dataset\default\0.0.0\f510e7ced81c9af0aecc4648f5fe470b3c0132c7\defactify_text_dataset-test.arrow
 - Rajarshi-Roy-research___defactify_text_dataset\default\0.0.0\f510e7ced81c9af0aecc4648f5fe470b3c0132c7\defactify_text_dataset-train.arrow
 - Rajarshi-Roy-research___defactify_text_dataset\default\0.0.0\f510e7ced81c9af0aecc4648f5fe470b3c0132c7\defactify_text_dataset-validation.arrow


In [48]:
from datasets import Dataset

def load_any(path):
    if path.endswith(".arrow"):
        return Dataset.from_file(path).to_pandas()
    return pd.read_parquet(path)

def find_split_file(files, split_name):
    matches = [f for f in files if split_name in os.path.basename(f).lower()]
    return matches[0] if matches else None

train_file = find_split_file(data_files, "train")
val_file = find_split_file(data_files, "valid")
test_file = find_split_file(data_files, "test")

if train_file:
    df_train = load_any(train_file)
else:
    df_train = pd.concat([load_any(f) for f in data_files], ignore_index=True)

df_val = load_any(val_file) if val_file else None
df_test = load_any(test_file) if test_file else None

print("Train:", df_train.shape)
print("Val:  ", df_val.shape if df_val is not None else "not found")
print("Test: ", df_test.shape if df_test is not None else "not found")

df_train.head()

Train: (51247, 3)
Val:   (10983, 3)
Test:  (10963, 3)


,Text,Label_A,Label_B
0,Comments\nThe U.S. bombings thatended World War II didn’t mark theclose of atomic warfare.They were just the beginni...,0,Human_Story
1,## The Unseen Scars: The Enduring Human Toll of Nuclear Testing\n\n**By [Your Name]**\n\n**Published: [Date]**\n\nTh...,1,Gemma-2-9B
2,\n\nTitle: The Atomic Aftermath: The U.S. Bombings of World War II and the Dawn of Nuclear Warfare\n\nThe U.S. bombi...,1,Mistral-7B
3,"Human Toll of Nuclear Testing: Unraveling the Invisible LegacyIn the atomic age, the quest for nuclear prowess has l...",1,Qwen-2-72B
4,The Ghostly Whispers of Trinity The Enduring Human Toll of Nuclear Testing\n\nBy Your Name\n\nSeventyeight years ag...,1,Llama-8B


In [49]:
# If val/test weren't found as separate files, carve them out of df_train now
if df_val is None or df_test is None:
    from sklearn.model_selection import train_test_split
    df_train, df_temp = train_test_split(df_train, test_size=0.30, stratify=df_train["Label_A"], random_state=42)
    df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp["Label_A"], random_state=42)
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

print("Train:", df_train.shape, "Val:", df_val.shape, "Test:", df_test.shape)


Train: (51247, 3) Val: (10983, 3) Test: (10963, 3)


## 3. Quick Recap EDA

In [50]:
print(df_train["Label_A"].value_counts())
print()
print(df_train["Label_B"].value_counts())


Label_A
1    43926
0     7321
Name: count, dtype: int64

Label_B
Human_Story    7321
Gemma-2-9B     7321
Mistral-7B     7321
Qwen-2-72B     7321
Llama-8B       7321
Yi-Large       7321
GPT-4o         7321
Name: count, dtype: int64


## 4. Baseline Detector — TF-IDF + Logistic Regression

Fast, CPU-friendly, and a solid sanity-check oracle before investing in a transformer classifier. `Label_A`: 0 = human, 1 = AI-generated.


In [51]:
# Clean missing/empty text before any downstream processing
print("Before cleaning:")
print("Train NaNs:", df_train["Text"].isna().sum(), "| Val NaNs:", df_val["Text"].isna().sum())

df_train = df_train.dropna(subset=["Text"]).reset_index(drop=True)
df_val = df_val.dropna(subset=["Text"]).reset_index(drop=True)
df_test = df_test.dropna(subset=["Text"]).reset_index(drop=True)

# Also drop empty/whitespace-only strings, which pass isna() but still break vectorizers
df_train = df_train[df_train["Text"].str.strip() != ""].reset_index(drop=True)
df_val = df_val[df_val["Text"].str.strip() != ""].reset_index(drop=True)
df_test = df_test[df_test["Text"].str.strip() != ""].reset_index(drop=True)

print("\nAfter cleaning:")
print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

Before cleaning:
Train NaNs: 66 | Val NaNs: 19

After cleaning:
Train: (51181, 3)
Val:   (10964, 3)
Test:  (10963, 3)


In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")

X_train = vectorizer.fit_transform(df_train["Text"])
X_val = vectorizer.transform(df_val["Text"])

y_train = df_train["Label_A"]
y_val = df_val["Label_A"]

clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
val_probs = clf.predict_proba(X_val)[:, 1]

print(classification_report(y_val, val_preds, target_names=["Human", "AI"]))
print("ROC-AUC:", roc_auc_score(y_val, val_probs))


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


              precision    recall  f1-score   support

       Human       1.00      0.94      0.97      1561
          AI       0.99      1.00      0.99      9403

    accuracy                           0.99     10964
   macro avg       0.99      0.97      0.98     10964
weighted avg       0.99      0.99      0.99     10964

ROC-AUC: 0.9997768782204052


In [53]:
def detector_score(texts):
    """Returns P(AI-generated) for a list of texts using the TF-IDF baseline detector."""
    X = vectorizer.transform(texts)
    return clf.predict_proba(X)[:, 1]

# Sanity check
sample_texts = df_val["Text"].iloc[:3].tolist()
print(detector_score(sample_texts))


[0.96860738 0.98670918 0.98830697]


## 5. Detector v2 — Fine-Tuned DistilBERT (Optional, GPU Recommended)

Skip this section on CPU-only machines and use the TF-IDF detector from Section 4 as your oracle instead. Run this if you have a GPU and want a stronger, more robust detector for evaluating your humanizer later.


In [54]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_NAME = "distilbert-base-uncased"
det_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return det_tokenizer(batch["Text"], truncation=True, padding="max_length", max_length=256)

train_ds = Dataset.from_pandas(df_train[["Text", "Label_A"]].rename(columns={"Label_A": "labels"}))
val_ds = Dataset.from_pandas(df_val[["Text", "Label_A"]].rename(columns={"Label_A": "labels"}))

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)


Device: cpu


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Map: 100%|██████████| 10964/10964 [00:05<00:00, 2004.09 examples/s]


In [ ]:
det_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./detector_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=det_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4284.14it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set 

Epoch,Training Loss,Validation Loss


## 6. Humanizer — T5 Seq2Seq Fine-Tuning Skeleton

**Key gap to resolve before running this for real:** this dataset is not row-paired (AI text isn't linked to the specific human text it should map to). The cell below gives you the full training skeleton with a `build_pairs()` function you need to adapt based on what pairing strategy you settle on:

- **If you find shared article/prompt IDs** in the raw files (check columns beyond `Text`/`Label_A`/`Label_B` — inspect `df_train.columns` and any metadata files in your dataset folder), use those to build true `(AI_text, human_text)` pairs.
- **If no pairing is recoverable**, the placeholder below uses a random-matching approximation as a functional stand-in — expect weaker results, and prioritize sourcing a genuinely paired dataset (e.g. build your own using DIPPER-paraphrased human text, which is paired by construction).


In [ ]:
print("Available columns:", df_train.columns.tolist())
# Inspect for any hidden ID/prompt columns that could enable true pairing
df_train.head(3)


In [ ]:
def build_pairs(df, n_pairs=5000, random_state=42):
    """
    Builds (AI_text -> human_text) training pairs.
    Currently uses random matching as a placeholder — replace with ID-based
    matching once you've confirmed a shared key between human and AI rows.
    """
    human_df = df[df["Label_A"] == 0].sample(n=n_pairs, random_state=random_state, replace=True).reset_index(drop=True)
    ai_df = df[df["Label_A"] == 1].sample(n=n_pairs, random_state=random_state, replace=True).reset_index(drop=True)

    pairs = pd.DataFrame({
        "input_text": "humanize: " + ai_df["Text"],
        "target_text": human_df["Text"],
    })
    return pairs

train_pairs = build_pairs(df_train, n_pairs=5000)
val_pairs = build_pairs(df_val, n_pairs=1000)

train_pairs.head()


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

T5_MODEL = "t5-small"
t5_tokenizer = T5Tokenizer.from_pretrained(T5_MODEL)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_MODEL).to(device)

MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256

def tokenize_pairs(batch):
    model_inputs = t5_tokenizer(
        batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length"
    )
    labels = t5_tokenizer(
        batch["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_pairs_ds = Dataset.from_pandas(train_pairs)
val_pairs_ds = Dataset.from_pandas(val_pairs)

train_pairs_ds = train_pairs_ds.map(tokenize_pairs, batched=True, remove_columns=train_pairs.columns.tolist())
val_pairs_ds = val_pairs_ds.map(tokenize_pairs, batched=True, remove_columns=val_pairs.columns.tolist())


In [ ]:
humanizer_args = TrainingArguments(
    output_dir="./humanizer_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=50,
    predict_with_generate=True,
) if False else TrainingArguments(   # predict_with_generate requires Seq2SeqTrainingArguments — see next cell
    output_dir="./humanizer_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=50,
)


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tokenizer, model=t5_model)

seq2seq_args = Seq2SeqTrainingArguments(
    output_dir="./humanizer_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=50,
    predict_with_generate=True,
)

seq2seq_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=seq2seq_args,
    train_dataset=train_pairs_ds,
    eval_dataset=val_pairs_ds,
    data_collator=data_collator,
)

seq2seq_trainer.train()


In [ ]:
def humanize(text, max_length=256):
    input_text = "humanize: " + text
    inputs = t5_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(device)
    outputs = t5_model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try it on a sample AI-generated text
sample_ai_text = df_val[df_val["Label_A"] == 1]["Text"].iloc[0]
print("--- ORIGINAL (AI) ---")
print(sample_ai_text[:400])
print()
print("--- HUMANIZED ---")
print(humanize(sample_ai_text))


## 7. Evaluation — Detector-as-Oracle

Feed humanized outputs back through the Section 4 detector. A successful humanizer should lower `detector_score` (P(AI)) relative to the original AI text, while ideally preserving meaning (add a semantic-similarity check, e.g. cosine similarity of sentence embeddings, before trusting this metric alone).


In [ ]:
eval_sample = df_val[df_val["Label_A"] == 1]["Text"].sample(20, random_state=42).tolist()

original_scores = detector_score(eval_sample)
humanized_texts = [humanize(t) for t in eval_sample]
humanized_scores = detector_score(humanized_texts)

results = pd.DataFrame({
    "original_ai_score": original_scores,
    "humanized_score": humanized_scores,
    "delta": humanized_scores - original_scores,
})

print(results.describe())
results.head()


## 8. Save Artifacts

In [ ]:
os.makedirs("models/humanizer_t5", exist_ok=True)
os.makedirs("models/detector_tfidf", exist_ok=True)

t5_model.save_pretrained("models/humanizer_t5")
t5_tokenizer.save_pretrained("models/humanizer_t5")

import joblib
joblib.dump(vectorizer, "models/detector_tfidf/vectorizer.joblib")
joblib.dump(clf, "models/detector_tfidf/classifier.joblib")

print("Saved humanizer model to models/humanizer_t5/")
print("Saved detector baseline to models/detector_tfidf/")


## Next Steps

- **Resolve pairing (Section 6):** the random-matched pairs are a functional placeholder, not real training signal. Prioritize either (a) recovering true prompt-level pairing from the source files, or (b) building your own paired corpus via DIPPER-style paraphrasing of the human-text subset here.
- **Strengthen the oracle:** swap the TF-IDF detector for the fine-tuned DistilBERT from Section 5, or bring in a public detector (RoBERTa-based, trained on RAID/MAGE) so your evaluation isn't just measuring whether you've overfit to your own classifier's blind spots.
- **Add a semantic-preservation check** (e.g. sentence-transformers cosine similarity) to Section 7 — a humanizer that reduces detector score by making the text incoherent isn't actually useful.
- **Scale up the base model** (t5-base/t5-large or a BART variant) once the pipeline is validated end-to-end on t5-small.
